<a href="https://colab.research.google.com/github/paulaprado1904/AlgoritmoEvolutivo/blob/main/AE_Es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math
import time
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# Contador global de avaliações completas da função objetivo.
# É usado como medida de custo computacional e como critério de parada.
GLOBAL_AVALIACOES = 0


# ======================================================
# === Estrutura do indivíduo (representação absoluta) ===
# ======================================================
class Individuo:
    """
    Representa uma solução candidata para o problema de dobramento proteico
    no modelo HP em malha 3D.

    Cada indivíduo é codificado por uma sequência de movimentos absolutos
    em uma malha cúbica tridimensional. A partir dessa sequência, o algoritmo
    reconstrói as coordenadas da cadeia e calcula as métricas usadas na avaliação.
    """
    def __init__(self, movimentos):
        # Cromossomo do indivíduo: sequência de movimentos absolutos
        self.movimentos = movimentos[:]

        # Coordenadas reconstruídas da cadeia
        self.coords = None

        # Número de colisões
        self.nC = 0

        # Número de contatos H-H não consecutivos
        self.nH = 0

        # Valor da função objetivo principal
        self.f = None

        # Métricas auxiliares de compactação
        self.dmax = None
        self.davg = None

        # Campo mantido por compatibilidade com outras variantes
        self.delta = None

        # Energia simplificada baseada nas distâncias entre resíduos H
        self.energiaSimplif = None

        # Energia equivalente da solução
        self.energia = None

    def __str__(self):
        """
        Retorna uma representação textual resumida do indivíduo,
        útil para depuração e inspeção manual.
        """
        f_val = self.f if self.f is not None else 0
        d_max = self.dmax if self.dmax is not None else 0
        d_avg = self.davg if self.davg is not None else 0
        delta_val = self.delta if self.delta is not None else 0
        energia_simplif_val = self.energiaSimplif if self.energiaSimplif is not None else 0
        energia_val = self.energia if self.energia is not None else 0

        return (
            f"f(c): {f_val:3} | "
            f"nC: {self.nC:2} | "
            f"dmax: {d_max:5.2f} | "
            f"davg: {d_avg:5.2f} | "
            f"delta: {delta_val:6.2f} | "
            f"energiaSimplif: {energia_simplif_val:6.2f} | "
            f"Energia: {energia_val}"
        )


# ======================================================
# === Movimentos absolutos em 3D ===
# ======================================================
def delta_from_move_3d(mov):
    """
    Converte o código de um movimento em seu deslocamento cartesiano
    correspondente na malha 3D.

    Convenção adotada:
    0 -> +x
    1 -> +y
    2 -> +z
    3 -> -z
    4 -> -y
    5 -> -x
    """
    if mov == 0:   # U: +x
        return (1, 0, 0)
    if mov == 1:   # L: +y
        return (0, 1, 0)
    if mov == 2:   # F: +z
        return (0, 0, 1)
    if mov == 3:   # B: -z
        return (0, 0, -1)
    if mov == 4:   # R: -y
        return (0, -1, 0)
    if mov == 5:   # D: -x
        return (-1, 0, 0)
    raise ValueError("mov inválido (esperado 0..5)")


# ======================================================
# === Construção da conformação e contagem de colisões ===
# ======================================================
def construir_coordenadas(ind):
    """
    Reconstrói as coordenadas da cadeia a partir da sequência de movimentos
    do indivíduo.

    Retorna:
    - coords: lista de coordenadas ocupadas pela cadeia
    - nC: número de colisões detectadas

    Nesta versão, nC é contado pelo número de posições ocupadas mais de uma vez.
    """
    x = y = z = 0
    coords = [(0, 0, 0)]
    ocupacao = {(0, 0, 0): 1}

    for mov in ind.movimentos:
        dx, dy, dz = delta_from_move_3d(mov)
        x += dx
        y += dy
        z += dz
        pt = (x, y, z)
        coords.append(pt)
        ocupacao[pt] = ocupacao.get(pt, 0) + 1

    nC = sum(1 for v in ocupacao.values() if v > 1)

    ind.coords = coords
    ind.nC = nC
    return coords, nC


def avaliar_base(ind, cadeia, rho=1.0):
    """
    Avalia completamente um indivíduo.

    Métricas calculadas:
    - nC: número de colisões
    - nH: número de contatos H-H não consecutivos
    - f(c): função objetivo base
    - dmax: maior distância entre pares H-H válidos
    - davg: média das distâncias entre pares H-H válidos
    - energiaSimplif: soma das distâncias entre resíduos H
    - energia: energia equivalente da solução

    Regras:
    - Se houver colisão, a solução é tratada como inviável.
    - A energia simplificada recebe uma penalização muito alta em soluções com colisão.
    - Se não houver colisão, f(c) = nH e energia = -f.
    """
    global GLOBAL_AVALIACOES
    GLOBAL_AVALIACOES += 1

    PENAL = 1e6

    coords, nC = construir_coordenadas(ind)
    n = len(cadeia)

    # Salva as coordenadas reconstruídas no indivíduo
    ind.coords = coords

    # --------------------------------------------------
    # Distâncias entre resíduos H
    # --------------------------------------------------
    H_idx = [i for i, a in enumerate(cadeia) if a == "H"]
    Es = 0.0
    dists = []
    m = len(H_idx)

    for a in range(m):
        for b in range(a + 1, m):
            p1 = coords[H_idx[a]]
            p2 = coords[H_idx[b]]

            # Ignora pares que colapsaram na mesma posição devido a colisão
            if p1 == p2:
                continue

            d = math.sqrt(
                (p1[0] - p2[0]) ** 2 +
                (p1[1] - p2[1]) ** 2 +
                (p1[2] - p2[2]) ** 2
            )

            dists.append(d)

            i = H_idx[a]
            j = H_idx[b]

            # Ignora pares consecutivos na cadeia ao somar Es
            if abs(i - j) == 1:
                continue

            Es += d

    if not dists:
        dmax = float('inf')
        davg = float('inf')
    else:
        dmax = max(dists)
        davg = sum(dists) / len(dists)

    # --------------------------------------------------
    # Caso inviável: há colisões
    # --------------------------------------------------
    if nC > 0:
        ind.nC = nC
        ind.nH = 0
        ind.f = -rho * nC

        # Mantém dmax e davg sem penalização explícita adicional
        ind.dmax = dmax
        ind.davg = davg

        ind.delta = None
        ind.energia = -ind.f

        # Penalização forte na energia simplificada
        Es = Es + (PENAL * nC)
        ind.energiaSimplif = Es
        return

    # --------------------------------------------------
    # Caso viável: sem colisões
    # Conta contatos H-H não consecutivos e adjacentes no espaço
    # --------------------------------------------------
    nH = 0
    for i in range(n):
        if cadeia[i] != "H":
            continue
        for j in range(i + 2, n):
            if cadeia[j] != "H":
                continue

            pi = coords[i]
            pj = coords[j]

            # Distância Manhattan entre os resíduos i e j
            manh = (
                abs(pi[0] - pj[0]) +
                abs(pi[1] - pj[1]) +
                abs(pi[2] - pj[2])
            )

            if manh == 1:
                nH += 1

    ind.nC = 0
    ind.nH = nH
    ind.f = nH
    ind.dmax = dmax
    ind.davg = davg
    ind.delta = None
    ind.energiaSimplif = Es
    ind.energia = -ind.f


# ======================================================
# === Geração de indivíduo aleatório ===
# ======================================================
def gerar_individuo_aleatorio(n_residuos):
    """
    Gera um indivíduo aleatório com n_residuos - 1 movimentos.

    Nesta implementação, todos os 6 movimentos possíveis podem aparecer
    na geração inicial.
    """
    L = n_residuos - 1
    movimentos = [random.randint(0, 5) for _ in range(L)]
    return Individuo(movimentos)


# ======================================================
# === Recombinação por múltiplos pontos ===
# ======================================================
def crossover_multipontos_2filhos(pai, mae, n_residuos):
    """
    Gera dois filhos por crossover de múltiplos pontos.

    O número de cortes é proporcional ao tamanho da proteína.
    """
    L = len(pai.movimentos)
    if L != len(mae.movimentos):
        raise ValueError("Pais com comprimentos diferentes de cromossomo")

    num_cortes = int(round(n_residuos / 10.0))
    num_cortes = max(1, min(num_cortes, L - 1))
    cortes = sorted(random.sample(range(1, L), num_cortes))

    def build_child(start_with_pai: bool):
        filho_mov = []
        pais = [pai.movimentos, mae.movimentos]
        idx = 0 if start_with_pai else 1
        ini = 0
        for c in cortes + [L]:
            filho_mov.extend(pais[idx][ini:c])
            idx = 1 - idx
            ini = c
        return Individuo(filho_mov)

    return build_child(True), build_child(False)


# ======================================================
# === Mutação gene a gene ===
# ======================================================
def mutacao_por_gene(ind, taxa_mut):
    """
    Aplica mutação gene a gene.

    Cada posição do cromossomo é alterada com probabilidade taxa_mut,
    sendo substituída por um novo movimento aleatório.
    """
    novo = Individuo(ind.movimentos[:])
    for j in range(len(novo.movimentos)):
        if random.random() <= taxa_mut:
            novo.movimentos[j] = random.randint(0, 5)
    return novo


# ======================================================
# === Seleção e sobrevivência baseadas em energia simplificada ===
# ======================================================
def selecao_torneio_mono(pop, k=3):
    """
    Seleção por torneio monoobjetivo.

    Nesta variante, o critério principal é minimizar a energia simplificada.
    """
    if not pop:
        raise RuntimeError("População vazia no torneio.")

    cand = random.sample(pop, k=min(k, len(pop)))
    return min(cand, key=lambda ind: ind.energiaSimplif)


def selecao_sobreviventes_mu_plus_lambda(pop, filhos, pop_size, elitismo=1):
    """
    Seleção de sobreviventes no esquema (mu + lambda).

    Estratégia:
    - preserva uma elite da população anterior;
    - combina pais e filhos;
    - remove duplicatas por cromossomo;
    - mantém os melhores indivíduos segundo energiaSimplif.
    """
    elite = sorted(pop, key=lambda ind: ind.energiaSimplif)[:max(0, elitismo)]

    candidatos = pop + filhos

    vistos = set()
    unicos = []
    for ind in sorted(candidatos, key=lambda ind: ind.energiaSimplif):
        key = tuple(ind.movimentos)
        if key not in vistos:
            vistos.add(key)
            unicos.append(ind)

    nova = []
    elite_keys = {tuple(e.movimentos) for e in elite}
    for e in elite:
        nova.append(e)

    for ind in unicos:
        if len(nova) >= pop_size:
            break
        if tuple(ind.movimentos) in elite_keys:
            continue
        nova.append(ind)

    while len(nova) < pop_size:
        nova.append(random.choice(candidatos))

    return nova


# ======================================================
# === Geração de gráfico da execução ===
# ======================================================
def salvar_grafico_melhor_fitness_com_marcador(df_hist, pasta_saida, nome_proteina, execucao_id):
    """
    Salva um gráfico da evolução do melhor fitness ao longo das gerações,
    destacando a geração em que ocorreu o melhor valor.
    """
    if df_hist is None or df_hist.empty:
        print("[WARN] df_hist vazio.")
        return None

    if "Iteracao" not in df_hist.columns or "Melhor Fitness" not in df_hist.columns:
        print("[WARN] df_hist sem colunas necessárias (Iteracao, Melhor Fitness).")
        return None

    xs = df_hist["Iteracao"].to_numpy()
    ys = df_hist["Melhor Fitness"].to_numpy()

    idx_best = int(df_hist["Melhor Fitness"].idxmax())
    best_gen = int(df_hist.loc[idx_best, "Iteracao"])
    best_fit = float(df_hist.loc[idx_best, "Melhor Fitness"])

    plt.figure(figsize=(8, 4.5))
    plt.plot(xs, ys)
    plt.axvline(best_gen, linestyle="--", linewidth=1)
    plt.scatter([best_gen], [best_fit], s=60)

    plt.xlabel("Geração")
    plt.ylabel("Melhor fitness (nH)")
    plt.title(f"{nome_proteina} — exec {execucao_id} — melhor gen {best_gen} (fit={best_fit:.0f})")
    plt.grid(True, alpha=0.3)

    ultima_gen = int(df_hist["Iteracao"].max())
    nome_arquivo = f"{nome_proteina}_exec{execucao_id:02d}_ate{ultima_gen:04d}_bestgen{best_gen:04d}.png"
    caminho_fig = os.path.join(pasta_saida, nome_arquivo)

    plt.tight_layout()
    plt.savefig(caminho_fig, dpi=200)
    plt.close()
    return caminho_fig


# ======================================================
# === Regra de mutação condicional ===
# ======================================================
def precisa_mutar(ind, melhor):
    """
    Retorna True se o indivíduo não for melhor que a referência.

    Nesta implementação, a comparação está baseada em f, e não diretamente
    em energiaSimplif.
    """
    if melhor is None:
        return False
    return ind.f <= melhor.f


# ======================================================
# === Algoritmo Evolutivo Monoobjetivo com energia simplificada ===
# ======================================================
def aemo_hp_3d_sem_mc(
    cadeia_raw,
    pop_init=8,
    taxa_mut=0.2,
    rho=1.0,
    torneio_k=2,
    taxa_cross=1.0,
    max_avaliacoes=None,
    seed=None,
    elitismo=1,
    target_nH=None
):
    """
    Executa o algoritmo evolutivo monoobjetivo com foco em energia simplificada.

    Estrutura geral:
    1. Gera população inicial aleatória
    2. Avalia os indivíduos
    3. Seleciona pais por torneio com base em energiaSimplif
    4. Gera filhos por crossover
    5. Aplica mutação condicional
    6. Seleciona sobreviventes no esquema (mu + lambda)
    7. Atualiza estatísticas até atingir o critério de parada

    Critérios de parada:
    - atingir o número máximo de avaliações;
    - atingir um alvo de nH, quando informado.
    """
    global GLOBAL_AVALIACOES

    cadeia = ''.join([c for c in cadeia_raw.strip().upper() if c in ('H', 'P')])
    n_res = len(cadeia)
    if n_res < 2:
        raise ValueError("Cadeia muito curta")

    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # ---------------- População inicial ----------------
    populacao = []
    for _ in range(pop_init):
        if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
            break
        ind = gerar_individuo_aleatorio(n_res)
        avaliar_base(ind, cadeia, rho=rho)
        populacao.append(ind)

    if not populacao:
        raise RuntimeError("Nenhum indivíduo pôde ser avaliado (max_avaliacoes muito baixo).")

    # Melhor indivíduo global segundo o critério atualmente usado no código
    melhor_global = max(populacao, key=lambda ind: ind.f)
    geracao_melhor = 0
    aval_ate_target = None

    # Histórico da execução
    melhores_f = []
    medias_f = []
    desvios_f = []

    geracao = 0
    while True:
        if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
            break

        if target_nH is not None and melhor_global.nC == 0 and melhor_global.nH >= target_nH:
            if aval_ate_target is None:
                aval_ate_target = GLOBAL_AVALIACOES
            break

        # ---------- Geração de filhos ----------
        filhos = []
        while len(filhos) < pop_init:
            pai = selecao_torneio_mono(populacao, k=torneio_k)
            mae = selecao_torneio_mono(populacao, k=torneio_k)

            if random.random() < taxa_cross:
                f1, f2 = crossover_multipontos_2filhos(pai, mae, n_res)
            else:
                f1, f2 = Individuo(pai.movimentos[:]), Individuo(mae.movimentos[:])

            for filho, ref in ((f1, pai), (f2, mae)):
                if len(filhos) >= pop_init:
                    break

                avaliar_base(filho, cadeia, rho=rho)

                # Aplica mutação apenas se o filho não superar a referência
                if precisa_mutar(filho, ref):
                    filho = mutacao_por_gene(filho, taxa_mut)
                    avaliar_base(filho, cadeia, rho=rho)

                filhos.append(filho)

        # ---------- Seleção de sobreviventes ----------
        populacao = selecao_sobreviventes_mu_plus_lambda(
            populacao, filhos, pop_size=pop_init, elitismo=elitismo
        )

        # ---------- Estatísticas da geração ----------
        f_vals = np.array([ind.f for ind in populacao], dtype=float)
        medias_f.append(float(np.mean(f_vals)))
        desvios_f.append(float(np.std(f_vals)))

        melhor_ger = max(populacao, key=lambda ind: ind.f)
        melhores_f.append(melhor_ger.f)

        if melhor_ger.f > melhor_global.f:
            melhor_global = melhor_ger
            geracao_melhor = geracao + 1

        geracao += 1

    resumo = {
        "Melhor Fitness (nH penalizado se colisão)": melhor_global.f,
        "nH(c)": melhor_global.nH,
        "nC(c)": melhor_global.nC,
        "Geração do Melhor": geracao_melhor,
        "Aval_ate_target": aval_ate_target,
        "Aval_total": GLOBAL_AVALIACOES,
    }

    df_hist = pd.DataFrame({
        "Iteracao": list(range(len(melhores_f))),
        "Fitness Médio": medias_f,
        "Desvio Padrão": desvios_f,
        "Melhor Fitness": melhores_f
    })

    return df_hist, resumo, melhor_global, populacao


# ======================================================
# === Execução repetida dos experimentos ===
# ======================================================
if __name__ == "__main__":
    # Arquivo da proteína e diretório de saída
    caminho_arquivo = "/content/drive/MyDrive/Proteínas TCC/273d.10.txt"
    pasta_saida = "/content/drive/MyDrive/Proteínas TCC/resultados AEMO Es - Chris/273d10"
    os.makedirs(pasta_saida, exist_ok=True)

    # Leitura da instância:
    # linha 1 -> número de resíduos
    # linha 2 -> cadeia HP
    with open(caminho_arquivo) as f:
        n = int(f.readline().strip())
        cadeia = ''.join([c for c in f.readline().strip().upper() if c in ('H', 'P')])
        assert len(cadeia) == n

    nome_proteina = os.path.splitext(os.path.basename(caminho_arquivo))[0]

    # Ótimo conhecido da instância
    ENERGIA_OTIMA = -11

    if n <= 36:
        # Parâmetros experimentais para proteínas pequenas
        taxa_mut = 0.03
        taxa_cross = 0.80
        MAX_AVALIACOES = 1_500_000

    NUM_EXECUCOES = 50
    resultados_execucoes = []
    hists_execucoes = []

    for i in range(NUM_EXECUCOES):
        GLOBAL_AVALIACOES = 0
        random.seed()
        np.random.seed()

        print(f"\n========== EXECUÇÃO {i+1}/{NUM_EXECUCOES} ==========")

        # Como energia = -nH em soluções factíveis, o alvo em nH é o oposto da energia ótima
        NH_OTIMO = -ENERGIA_OTIMA

        df_hist, resumo, melhor, pop_final = aemo_hp_3d_sem_mc(
            cadeia,
            pop_init=120,
            taxa_mut=taxa_mut,
            rho=1.0,
            torneio_k=3,
            taxa_cross=taxa_cross,
            max_avaliacoes=MAX_AVALIACOES,
            elitismo=0,
            target_nH=NH_OTIMO,
        )

        # Estatísticas da população final desta execução
        gen_unicos_final = len({tuple(ind.movimentos) for ind in pop_final})
        E_final = np.array([ind.energia for ind in pop_final], dtype=float)
        E_media_final = float(np.mean(E_final))
        E_dp_final = float(np.std(E_final))

        caminho_fig = salvar_grafico_melhor_fitness_com_marcador(
            df_hist, pasta_saida, nome_proteina, i + 1
        )
        print(f"[OK] Execução {i+1}: gráfico salvo em {caminho_fig}")

        hists_execucoes.append(df_hist)

        # Energia equivalente só faz sentido para solução factível
        melhor_energia = (-melhor.nH) if melhor.nC == 0 else float("inf")

        hit_otimo = (
            (resumo["Aval_ate_target"] is not None) and
            (melhor.nC == 0) and
            (melhor.nH >= NH_OTIMO)
        )
        aval_ate_otimo = resumo["Aval_ate_target"]

        resultados_execucoes.append({
            "Execucao": i + 1,
            "Melhor_Energia": melhor_energia,
            "Hit_otimo": hit_otimo,
            "Aval_ate_otimo": aval_ate_otimo,
            "Aval_total": resumo["Aval_total"],
            "Melhor_fitness": resumo["Melhor Fitness (nH penalizado se colisão)"],
            "nH": melhor.nH,
            "nC": melhor.nC,
            "Genotipos_unicos_final": gen_unicos_final,
            "Energia_media_finalpop": E_media_final,
            "Energia_dp_finalpop": E_dp_final,
        })

    df_resumo = pd.DataFrame(resultados_execucoes)

    # ======================================================
    # === Estatísticas da melhor energia por execução ===
    # ======================================================
    energies = df_resumo["Melhor_Energia"].replace([np.inf, -np.inf], np.nan).dropna()

    melhor_energia_media = float(energies.mean()) if not energies.empty else float("nan")
    melhor_energia_dp = float(energies.std(ddof=1)) if len(energies) > 1 else float("nan")

    qtd_facteis = int(energies.shape[0])
    qtd_total = int(df_resumo.shape[0])

    print("\n\n===== RESUMO DAS 50 EXECUÇÕES =====")
    print(df_resumo)

    # Considera apenas execuções que atingiram o ótimo conhecido
    df_sucesso = df_resumo[df_resumo["Hit_otimo"] == True]

    if not df_sucesso.empty:
        media_aval = df_sucesso["Aval_ate_otimo"].mean()
        mediana_aval = df_sucesso["Aval_ate_otimo"].median()
    else:
        media_aval = float('nan')
        mediana_aval = float('nan')

    taxa_acertos = 100.0 * df_sucesso.shape[0] / NUM_EXECUCOES

    print("\n===== RESUMO GLOBAL (estilo artigo) =====")
    print(f"Ótimo conhecido: {ENERGIA_OTIMA}")
    print(f"Menor energia obtida: {df_resumo['Melhor_Energia'].min()}")
    print(f"Acurácia (Ac.): {taxa_acertos:.1f} %")
    print(f"Média Aval. (apenas execuções com ótimo): {media_aval:.0f}")
    print(f"Mediana Aval.: {mediana_aval:.0f}")

    # Salva resumo em arquivo texto
    caminho_saida_txt = os.path.join(pasta_saida, f"{nome_proteina}.txt")
    with open(caminho_saida_txt, "w") as f_out:
        f_out.write(f"Arquivo da proteína: {caminho_arquivo}\n")
        f_out.write(f"Nome da proteína (base): {nome_proteina}\n")
        f_out.write(f"Ótimo conhecido: {ENERGIA_OTIMA}\n\n")

        f_out.write("===== RESUMO DAS 50 EXECUÇÕES =====\n")
        f_out.write(df_resumo.to_string(index=False))
        f_out.write("\n\n===== RESUMO GLOBAL (estilo artigo) =====\n")
        f_out.write(f"Menor energia obtida: {df_resumo['Melhor_Energia'].min()}\n")
        f_out.write(f"Acurácia (Ac.): {taxa_acertos:.1f} %\n")
        f_out.write(f"Média Aval. (execuções com ótimo): {media_aval:.0f}\n")
        f_out.write(f"Mediana Aval.: {mediana_aval:.0f}\n")

        f_out.write("\n----- ESTATÍSTICAS (MELHOR ENERGIA POR EXECUÇÃO) -----\n")
        f_out.write(f"Melhor_Energia (média ± dp): {melhor_energia_media:.4f} ± {melhor_energia_dp:.4f}\n")
        f_out.write(f"Execuções factíveis consideradas: {qtd_facteis}/{qtd_total}\n")

    print(f"\nResumo salvo em: {caminho_saida_txt}")